# Granite Peak CRE: 03 Governance (runs before the pipeline)

Creates the Unity Catalog access model that the pipeline attaches to its gold tables:

- **Workspace groups:** portfolio leadership sees every fund; one analyst group per fund; a PII-reader group sees tenant contact details.
- **`fund_access`:** a mapping of group → fund (`*` = all funds).
- **`fund_row_filter`:** a row-filter function. `rent_roll` is published with `WITH ROW FILTER fund_row_filter ON (fund_id)`, so Genie, the SQL editor and dashboards all return only the funds a user is entitled to.
- **`mask_contact`:** a column-mask function on tenant contact fields in `tenant_directory`.
- **`abstract_review`:** the Delta table of analyst corrections to extracted lease fields. The app writes reviews to Lakebase, they're copied here, and the pipeline applies approved corrections in `lease_abstract`.

The pipeline references these functions, so this notebook must run first. The workspace groups are created with the Databricks SDK; this requires workspace-admin rights.

In [1]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
fq = f"`{catalog}`.`{schema}`"
print(f"Target: {catalog}.{schema}")

Target: smmorgan_catalog.granite_commercial_real_estate


In [2]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam

w = WorkspaceClient()
me = w.current_user.me()

GROUPS = {
    "granite_portfolio_leadership": "*",           # all funds
    "granite_fund_i_analysts":      "GP-FND-001",
    "granite_fund_ii_analysts":     "GP-FND-002",
    "granite_fund_iii_analysts":    "GP-FND-003",
    "granite_fund_iv_analysts":     "GP-FND-004",
    "granite_pii_readers":          None,          # unmasked tenant contact details
}

existing = {g.display_name: g for g in w.groups.list(attributes="id,displayName")}
for name in GROUPS:
    if name not in existing:
        existing[name] = w.groups.create(display_name=name)
        print(f"created group {name}")
    else:
        print(f"group exists  {name}")

# The build runs as its owner, who needs every fund to refresh the gold layer; contact details stay masked
leadership = existing["granite_portfolio_leadership"]
members = {m.value for m in (w.groups.get(leadership.id).members or [])}
if me.id not in members:
    w.groups.patch(leadership.id, operations=[iam.Patch(op=iam.PatchOp.ADD, path="members",
                                                         value=[{"value": me.id}])],
                   schemas=[iam.PatchSchema.URN_IETF_PARAMS_SCIM_API_MESSAGES_2_0_PATCH_OP])
    print("added the build owner to granite_portfolio_leadership")

group exists  granite_portfolio_leadership
group exists  granite_fund_i_analysts
group exists  granite_fund_ii_analysts
group exists  granite_fund_iii_analysts
group exists  granite_fund_iv_analysts
group exists  granite_pii_readers


In [3]:
rows = ", ".join(f"('{g}', '{f}')" for g, f in GROUPS.items() if f)
spark.sql(f"""
CREATE OR REPLACE TABLE {fq}.fund_access (
  group_name STRING COMMENT 'Workspace group',
  fund_id    STRING COMMENT 'Fund the group may see; * means every fund'
) COMMENT 'Fund entitlements used by fund_row_filter'
""")
spark.sql(f"INSERT INTO {fq}.fund_access VALUES {rows}")
display(spark.table(f"{fq}.fund_access"))

group_name                   | fund_id   
-----------------------------+-----------
granite_portfolio_leadership | *         
granite_fund_i_analysts      | GP-FND-001
granite_fund_ii_analysts     | GP-FND-002
granite_fund_iii_analysts    | GP-FND-003
granite_fund_iv_analysts     | GP-FND-004


In [4]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.fund_row_filter(p_fund_id STRING)
RETURNS BOOLEAN
COMMENT 'Row filter: true when the caller belongs to a group entitled to the fund (see fund_access)'
RETURN EXISTS (
  SELECT 1 FROM {fq}.fund_access a
  WHERE (a.fund_id = '*' OR a.fund_id = p_fund_id) AND is_member(a.group_name)
)
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {fq}.mask_contact(p_value STRING)
RETURNS STRING
COMMENT 'Column mask: tenant contact details are visible only to granite_pii_readers'
RETURN CASE WHEN is_member('granite_pii_readers') THEN p_value ELSE '[REDACTED]' END
""")

display(spark.sql(f"""
SELECT f.fund_id, {fq}.fund_row_filter(f.fund_id) AS caller_can_see,
       {fq}.mask_contact('contact1@example.com') AS masked_sample
FROM VALUES ('GP-FND-001'), ('GP-FND-002'), ('GP-FND-003'), ('GP-FND-004') AS f(fund_id)
"""))

fund_id    | caller_can_see | masked_sample
-----------+----------------+--------------
GP-FND-001 | True           | [REDACTED]   
GP-FND-002 | True           | [REDACTED]   
GP-FND-003 | True           | [REDACTED]   
GP-FND-004 | True           | [REDACTED]   


In [5]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {fq}.abstract_review (
  lease_id        STRING    COMMENT 'Lease whose abstract was reviewed',
  field_name      STRING    COMMENT 'Extracted field that was corrected, for example lease_end',
  extracted_value STRING    COMMENT 'Value produced by ai_extract',
  corrected_value STRING    COMMENT 'Value confirmed by the analyst',
  status          STRING    COMMENT 'approved or rejected',
  reviewer        STRING    COMMENT 'Reviewer identity',
  reviewed_at     TIMESTAMP COMMENT 'Review time'
) COMMENT 'Analyst corrections to extracted lease fields, copied from the Lakebase review table'
""")
print(f"abstract_review rows: {spark.table(f'{fq}.abstract_review').count()}")

abstract_review rows: 0
